# smart-commit DPO training on Kaggle

1. Settings: **Accelerator: GPU T4 x2**, **Internet: on** (needs a verified phone).
2. **Add Input** → the private dataset made by `sc-training kaggle-bundle dpo`.
3. **Save Version → Save & Run All (Commit)**: it runs in the background, up to 12 h.
4. Download `/kaggle/working/<run>.tar.gz` from the version's **Output**, `tar xzf` it
   into `training/runs/` and export with `export_gguf.py BASE out.gguf --adapter <SFT>
   --adapter runs/<run>/adapter` (the SFT adapter first, then the DPO one).

Runs go one per GPU, in the order of `runs.json`; each is packed as soon as it ends.

In [ ]:
import glob, json, os, pathlib, subprocess
bundle = pathlib.Path(glob.glob('/kaggle/input/**/runs.json', recursive=True)[0]).parent
spec = json.loads((bundle / 'runs.json').read_text())
print(bundle, json.dumps(spec, indent=1))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


In [ ]:
# Our locked versions; Kaggle's torch (and its CUDA) is kept. Kaggle's torchao is too old
# for this peft, which looks for it on every 16-bit layer, and nothing here uses it.
!pip uninstall -y -q torchao
!pip install -q -r {bundle}/requirements.txt
import torch, transformers, peft
print(torch.__version__, torch.version.cuda, transformers.__version__, peft.__version__)


In [ ]:
from huggingface_hub import snapshot_download
bases = {}
for run in spec['runs']:
    b = run['base']
    key = f"{b['repo']}@{b['revision']}"
    if key not in bases:
        bases[key] = snapshot_download(b['repo'], revision=b['revision'],
                                       local_dir=f"/kaggle/temp/{b['repo'].split('/')[1]}")
print(bases)


In [ ]:
import hashlib, time
work = pathlib.Path('/kaggle/working')
def sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()
gpus = torch.cuda.device_count()
per_gpu = spec.get('per_gpu', 1)
pending, running = list(spec['runs']), []
while pending or running:
    for gpu in range(gpus):
        while pending and sum(1 for g, _, _ in running if g == gpu) < per_gpu:
            run = pending.pop(0)
            assert sha256(bundle / run['train']) == run['train_sha256'], run['name']
            assert sha256(bundle / run['sft'] / 'adapter_model.safetensors') == run['sft_sha256'], run['name']
            out = work / 'runs' / run['name']
            out.mkdir(parents=True, exist_ok=True)
            log = open(work / f"{run['name']}.log", 'w')
            base = bases[f"{run['base']['repo']}@{run['base']['revision']}"]
            cmd = ['python', '-I', str(bundle / 'train_dpo.py'), str(bundle / run['train']), base,
                   str(bundle / run['sft']), str(out), '--val', str(bundle / run['val']), *run['args']]
            print(f"gpu {gpu}: {' '.join(cmd)}", flush=True)
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                    env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)})
            running.append((gpu, run, proc))
    time.sleep(30)
    for item in list(running):
        gpu, run, proc = item
        if proc.poll() is not None:
            running.remove(item)
            print(f"{run['name']}: exit {proc.returncode}", flush=True)
            print(open(work / f"{run['name']}.log").read()[-1500:], flush=True)
            # Packed right away, so a session that dies later keeps it.
            name = run['name']
            if (work / 'runs' / name / 'adapter').is_dir():
                subprocess.run(['tar', 'czf', str(work / f'{name}.tar.gz'), '-C', str(work / 'runs'),
                                f'{name}/adapter', f'{name}/run.json', '-C', str(work), f'{name}.log'],
                               check=True)

failed = [r['name'] for r in spec['runs'] if not (work / f"{r['name']}.tar.gz").exists()]
assert not failed, f'runs without an archive: {failed}'


In [ ]:
!ls -la {work}
